In [ ]:
from aneurysm_ai.api import *
from aneurysm_ai.data.datasets import build_vessel_samples,VesselDatasetFactory
from aneurysm_ai.data import DICOMSeries,NiftiDicomMapper,NiftiSegmentation,build_verified_mapping,split_series
from aneurysm_ai.data.aneurysm import AneurysmAnnotationBuilder,validate_annotations,build_positive_samples
from aneurysm_ai.data.roi import extract_roi,roi_to_25d,resize_25d
import numpy as np
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from torch.amp import autocast,GradScaler
from torch.utils.data import DataLoader
from aneurysm_ai.data.multiscale import MultiScaleAneurysmDataset
from aneurysm_ai.data.detection import AneurysmROIDataset
from aneurysm_ai.models import SwinVesselEncoder,VesselAwareFeatures,AneurysmFeatureEncoder,AneurysmDetector,ActorCritic,MultiScaleAneurysmDetector
from aneurysm_ai.training.aneurysm_trainer import AneurysmDetectionLoss
from aneurysm_ai.training import PPOTrainer,AneurysmSearchEnv
from aneurysm_ai.models.vessel import VesselModel
from aneurysm_ai.training.vessel_trainer import VesselTrainer

DATA_ROOT = (
    "/kaggle/input/competitions/"
    "rsna-intracranial-aneurysm-detection"
)

cfg = create_config(DATA_ROOT)
train_df = load_train_dataframe(cfg)
print(cfg)
print(check_dataset(cfg))
print(train_df.shape)

In [ ]:
uid = train_df.iloc[0]["SeriesInstanceUID"]
dicom = DICOMSeries(cfg.series_dir/str(uid))
print("Slices:", dicom.num_slices)
print("Shape:", dicom.shape)
print("Spacing:", dicom.spacing_xyz)

volume = dicom.load_volume()

print("Volume:", volume.shape)
print("----Checking Validation------")
point = np.array([
    dicom.columns / 2,
    dicom.rows / 2,
    dicom.num_slices / 2
])
world = dicom.voxel_to_world(point)
recovered = dicom.world_to_voxel(world)
print("Original :", point)
print("World    :", world)
print("Recovered:", recovered)

print(
    "Round-trip error:",
    np.linalg.norm(point - recovered)
)

In [ ]:
seg_files = list(cfg.segmentation_dir.glob("*.nii"))
print(f"Found -> {len(seg_files)}")
nifti_path = seg_files[0] # for testing purposes
mapper = NiftiDicomMapper(dicom)
information = mapper.inspect(nifti_path)
print(f"NifTI path -> {nifti_path}\n Information -> {information}\n")
segmentation = mapper.resample(nifti_path)
print(f"Segmentation shape -> {segmentation.shape}")
labels = np.unique(segmentation)
print("Labels -> ",labels)
assert set(labels).issubset(set(range(14)))
print("Mapping Validation: PASSED")

In [ ]:
z = dicom.num_slices // 2
plt.figure(figsize=(8,8))
plt.imshow(volume[1],cmap="gray")
mask = segmentation[z] > 0
plt.imshow(mask,alpha=0.4)
plt.axis("off")
plt.show()

In [ ]:
for seg in seg_files[:3]:
    seg = NiftiSegmentation(seg)
    print(f"Shape -> {seg.shape}\nSpacing -> {seg.spacing}\nLabels: {seg.validate_labels()}\nAffine -> {seg.affine}")

In [ ]:
verified_mapping = build_verified_mapping(
    series_dir=cfg.series_dir,
    segmentation_dir=cfg.segmentation_dir,
)

print("Verified pairs:", len(verified_mapping))

In [ ]:
for item in verified_mapping[:5]:
    print(item)
train_mapping,val_mapping = split_series(verified_mapping,val_fraction=0.15,seed=444)
train_samples,val_samples = build_vessel_samples(train_mapping,cfg.series_dir),build_vessel_samples(val_mapping,cfg.series_dir)
print("Train series:", len(train_mapping))
print("Val series:", len(val_mapping))
print("Train samples:", len(train_samples))
print("Val samples:", len(val_samples))
factory = VesselDatasetFactory(
    series_dir=cfg.series_dir,
    image_size=cfg.image_size,
    cache_dir=cfg.data_root / "cache" / "vessel"
)
train_dataset = factory.create(train_samples)
val_dataset = factory.create(val_samples)
print("Train: ",len(train_dataset))
print("Val: ",len(val_dataset))

In [ ]:
sample = train_dataset[0]
print("Image:",sample["image"].shape)
print("Mask:",sample["mask"].shape)
print("Image dtype:",sample["image"].dtype)
print("Mask dtype:",sample["mask"].dtype)
print("Mask labels:",torch.unique(sample["mask"]))

In [ ]:
image = sample["image"].numpy()
mask = sample["mask"].numpy()
plt.figure(figsize=(8, 8))
plt.imshow(image[1],cmap="gray")
plt.imshow(mask > 0,alpha=0.4)
plt.axis("off")
plt.show()

In [ ]:
device = cfg.device
model = VesselModel(model_name=cfg.swin_model,num_classes=cfg.segmentation_classes).to(device)
print("Parameters:",sum(p.numel() for p in model.parameters()) / 1e6,"M")

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=cfg.batch_size,shuffle=True,num_workers=cfg.num_workers,pin_memory=True,persistent_workers=cfg.num_workers > 0)
val_loader = DataLoader(val_dataset,batch_size=cfg.batch_size,shuffle=False,num_workers=cfg.num_workers,pin_memory=True,persistent_workers=cfg.num_workers > 0)
print(len(train_loader),len(val_loader))

In [ ]:
encoder_params,decoder_params = [],[]
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    if name.startswith("encoder."):
        encoder_params.append(parameter)
    else:
        decoder_params.append(parameter)
optimizer = torch.optim.AdamW([
        {
            "params": encoder_params,
            "lr": 1e-5,
        },
        {
            "params": decoder_params,
            "lr": 1e-4,
        },
    ],
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=cfg.epochs,eta_min=1e-6)
trainer = VesselTrainer(model,train_loader,val_loader,optimizer,device,epochs=cfg.epochs,scheduler=scheduler,checkpoint_dir=cfg.data_root/"checkpoints")
trainer.fit()

In [ ]:
model.eval()
batch = next(iter(val_loader))
with torch.no_grad():
    image = batch["image"].to(device)
    logits = model(image)
    prediction = logits.argmax(dim=1)
print("Prediction shape:",prediction.shape)
print("Prediction labels:",torch.unique(prediction).cpu().tolist())

In [ ]:
image = batch["image"][0, 1].cpu().numpy()
target = batch["mask"][0].cpu().numpy()
pred = prediction[0].cpu().numpy()
plt.figure(figsize=(8, 8))
plt.imshow(image,cmap="gray")
plt.imshow(pred > 0,alpha=0.4)
plt.axis("off")
plt.show()

In [ ]:
encoder = SwinVesselEncoder(model_name=cfg.swin_model,pretrained=True).to(device)
checkpoint = torch.load(
    cfg.data_root /
    "checkpoints" /
    "best_vessel.pt",
    map_location=device
)
state = checkpoint["model"]
encoder_state = {}
for key, value in state.items():
    if key.startswith("encoder."):
        new_key = key[len("encoder."):]
        encoder_state[new_key] = value
encoder.load_state_dict(encoder_state)
print("Loaded vessel-pretrained Swin.")
vessel_features = VesselAwareFeatures(channels=encoder.out_channels).to(device)

In [ ]:
batch = next(iter(train_loader))
images = batch["image"].to(device)
masks = batch["mask"].to(device)

with torch.no_grad():
    features = encoder(images)
    vessel_probability = (masks > 0).float().unsqueeze(1)
    enhanced_features, attention = (vessel_features(features,vessel_probability))
print("Input:",images.shape)
print("Swin:",features.shape)
print("Enhanced:",enhanced_features.shape)
print("Attention:",attention.shape)

In [ ]:
encoder.train()
vessel_features.train()
images = next(iter(train_loader))["image"].to(device)
masks = next(iter(train_loader))["mask"].to(device)
features = encoder(images)
mask = (masks > 0).float().unsqueeze(1)
enhanced, attention = (vessel_features(features,mask))
loss = enhanced.mean()
loss.backward()
print("Gradient check: PASSED")

In [ ]:
print("Encoder channels:", encoder.out_channels)
print("Feature shape:", features.shape)
print("Enhanced shape:", enhanced.shape)
print("Attention shape:", attention.shape)

assert features.shape[-2:] == (7, 7)
assert enhanced.shape == features.shape
assert attention.shape[1] == 1

print("Validation: PASSED")

In [ ]:
builder = AneurysmAnnotationBuilder(train_csv=cfg.train_csv,localizer_csv=cfg.localizer_csv,series_dir=cfg.series_dir)
annotations = builder.build()
print("Annotations:",len(annotations))
print(annotations.head())
errors = validate_annotations(annotations,cfg.series_dir)
print("Invalid annotations:",len(errors))

In [ ]:
positive_samples = build_positive_samples(annotations,cfg.series_dir,roi_size=(24,96,96))
sample = positive_samples[0]
dicom = DICOMSeries(cfg.series_dir /sample["series_uid"])
volume = dicom.load_volume()
roi = extract_roi(volume,sample["center"],roi_size=(24, 96, 96))
image_25d = roi_to_25d(roi)
image_25d = resize_25d(image_25d,size=224)
print("ROI:",roi.shape)
print("2.5D:",image_25d.shape)

In [ ]:
vessel_checkpoint = (
    cfg.data_root
    / "checkpoints"
    / "best_vessel.pt"
)
feature_encoder = AneurysmFeatureEncoder(model_name=cfg.swin_model,checkpoint_path=vessel_checkpoint).to(cfg.device)
print("Vessel-pretrained encoder loaded.")

In [ ]:
checkpoint = torch.load(
    vessel_checkpoint,
    map_location="cuda"
)
checkpoint_state = checkpoint["model"]
key = next(
    k for k in checkpoint_state
    if k.startswith("encoder.")
)
checkpoint_weight = checkpoint_state[key]
new_key = key[len("encoder."):]
loaded_weight = (feature_encoder.encoder.state_dict()[new_key])
difference = (checkpoint_weight - loaded_weight.cpu()).abs().max().item()
print("Maximum weight difference:",difference)

In [ ]:
feature_encoder.freeze_encoder()
trainable = sum(p.numel() for p in feature_encoder.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in feature_encoder.parameters() if not p.requires_grad)
print("Trainable: ",trainable / 1e6,"M")
print("Frozen: ",frozen / 1e6,"M")

In [ ]:
assert isinstance(feature_encoder,torch.nn.Module)
assert (feature_encoder.encoder.out_channels == 768)
assert all(not p.requires_grad for p in feature_encoder.encoder.parameters())
print("Transfer-learning validation: PASSED.")
feature_encoder.unfreeze_last_stage()
trainable_encoder = sum(p.numel() for p in feature_encoder.encoder.parameters() if p.requires_grad)
print("Final-stage trainable parameters: ",trainable_encoder / 1e6,"M")
feature_encoder.unfreeze_encoder()
print("All encoder parameters:",sum(p.numel() for p in feature_encoder.encoder.parameters() if p.requires_grad) / 1e6,"M")

In [ ]:
model = AneurysmDetector(model_name=cfg.swin_model,checkpoint_path=cfg.data_root/"checkpoints"/"best_vessel.pt",num_classes=cfg.num_anatomy_classes).to(cfg.device)

In [ ]:
model.freeze_encoder()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable -> {trainable}\nTotal -> {total}")

In [ ]:
images = batch["image"].to(cfg.device)
with torch.no_grad():
    predictions = model(images)
for key,value in predictions.items():
    if value is not None:
        print(f"{key}: {value.shape}")
criterion = AneurysmDetectionLoss()
B = images.shape[0]
objectness = torch.ones(B,1,device=cfg.device)
center = torch.full((B,3),0.5,device=cfg.device)
anatomy = torch.zeros(B,dtype=torch.long,device=cfg.device)
losses = criterion(predictions,objectness,center,anatomy)
for name,value in losses.items():
    print(f"{name}: {float(value)}")

In [ ]:
train_dataset = AneurysmROIDataset(samples=train_samples,series_dir=cfg.series_dir,roi_size=cfg.roi_medium,image_size=cfg.image_size)
val_dataset = AneurysmROIDataset(samples=val_samples,series_dir=cfg.series_dir,roi_size=cfg.roi_medium,image_size=cfg.image_size)
train_loader = DataLoader(train_dataset,cfg.batch_size,True,cfg.num_workers,pin_memory=True)
val_loader = DataLoader(val_dataset,cfg.batch_size,False,cfg.num_workers,pin_memory=True)
batch = next(iter(train_loader))
print(batch["image"].shape,batch["objectness"].shape,batch["center"].shape,batch["anatomy"].shape)
print("Positive: ",int(batch["objectness"].sum()))
print("Negative: ",int((batch["objectness"] == 0).sum()))
model = AneurysmDetector(cfg.swin_model,checkpoint_path=cfg.data_root/"checkpoints"/"best_vessel.pt",num_classes=cfg.num_anatomy_classes).to(cfg.device)
model.freeze_encoder()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,model.parameters()),lr=1e-4,weight_decay=1e-4)
criterion = AneurysmDetectionLoss(center_weight=2.0,anatomy_weight=1.0)
scaler = GradScaler("cuda")
model.train()
for batch in train_loader:
    images = batch["image"].to(cfg.device)
    objectness = batch["objectness"].to(cfg.device)
    center = batch["center"].to(cfg.device)
    anatomy = batch["anatomy"].to(cfg.device)
    optimizer.zero_grad(set_to_none=True)
    with autocast(device_type=cfg.device.type,enabled=cfg.device.type == "cuda"):
        predictions = model(images)
        losses = criterion(predictions,objectness,center,anatomy)
    scaler.scale(losses["total"]).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0,
    )
    scaler.step(optimizer)
    scaler.update()
    print(
        "loss:",
        float(losses["total"]),
        "obj:",
        float(losses["objectness"]),
        "center:",
        float(losses["center"]),
        "anat:",
        float(losses["anatomy"]),
    )
    break # Remove this break to train on the entire dataset. It's here for demonstration purposes.

In [ ]:
roi_sizes = {
    "coarse": cfg.roi_coarse,
    "medium": cfg.roi_medium,
    "fine": cfg.roi_fine,
}
dataset = MultiScaleAneurysmDataset(
    samples=train_samples,
    series_dir=cfg.series_dir,
    roi_sizes=roi_sizes,
    image_size=cfg.image_size,
)
sample = dataset[0]
print("coarse: ",sample["coarse"].shape)
print("medium:",sample["medium"].shape)
print("fine:",sample["fine"].shape)
print("center:",sample["center"])
model = MultiScaleAneurysmDetector(cfg.swin_model,checkpoint_path=(
        cfg.data_root
        / "checkpoints"
        / "best_vessel.pt"
    ),
    num_classes=cfg.anatomy_classes,
).to(cfg.device)
model.freeze_encoder()

In [ ]:
sample = dataset[0]
coarse = sample["coarse"].unsqueeze(0).to(cfg.device)
medium = sample["medium"].unsqueeze(0).to(cfg.device)
fine = sample["fine"].unsqueeze(0).to(cfg.device)
center = sample["center"].unsqueeze(0).to(cfg.device)
with torch.no_grad():
    output = model(
        coarse,
        medium,
        fine,
        center,
    )
for key, value in output.items():
    if value is not None:
        print(
            key,
            tuple(value.shape)
            if hasattr(value, "shape")
            else type(value),
        )

In [ ]:
# Testing purpose
vessel_mask = torch.rand(1,1,7,7,device=cfg.device)
output = model(coarse,medium,fine,center,vessel_mask)
print(f"Testing on random data -> {output}")
vessel_mask = torch.nn.functional.interpolate(vessel_mask,size=features.shape[-2:],mode="bilinear",align_corners=False)

In [ ]:
agent = ActorCritic(
    state_dim=cfg.rl_state_dim,
    action_dim=cfg.rl_action_dim,
    hidden_dim=128,
).to(cfg.device)

ppo = PPOTrainer(
    model=agent,
    lr=cfg.rl_lr,
    gamma=cfg.rl_gamma,
    gae_lambda=cfg.rl_gae_lambda,
    clip_eps=cfg.rl_clip_eps,
    value_coef=cfg.rl_value_coef,
    entropy_coef=cfg.rl_entropy_coef,
)

In [ ]:
# Testing the Environment
volume_shape = (
    200, 256, 256
)
aneurysm_centers = [
    [128, 120, 90],
]

env = AneurysmSearchEnv(
    volume_shape=volume_shape,
    aneurysm_centers=aneurysm_centers,
    max_steps=cfg.rl_max_steps,
    move_step=cfg.rl_move_step,
    scale_step=cfg.rl_scale_step,
)
state = env.reset(confidence=0.0)
print(state)

In [ ]:
state_tensor = torch.tensor(
    state,
    dtype=torch.float32,
    device=cfg.device,
).unsqueeze(0)

action, log_prob, value = (
    ppo.select_action(
        state_tensor
    )
)

print(
    "action:",
    int(action.item())
)

print(
    "value:",
    float(value.item())
)

In [ ]:
new_state, reward, done, info = (
    env.step(
        int(action.item()),
        new_confidence=0.2,
    )
)
print(
    "state:",
    new_state
)
print(
    "reward:",
    reward
)
print(
    "distance:",
    info["distance"]
)